<a href="https://colab.research.google.com/github/nicholasjayee/collab/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [2]:
# setting up uv
!uv init

Initialized project `content`


In [9]:
# imports
import sys

!uv add scikit-learn
!uv add pandas
!uv add tensorflow
!uv add torch
!uv add kagglehub
!uv add matplotlib


import kagglehub

Resolved 80 packages in 1ms
Checked 78 packages in 1ms
Resolved 80 packages in 1ms
Checked 78 packages in 1ms
Resolved 80 packages in 1ms
Checked 78 packages in 1ms
Resolved 80 packages in 1ms
Checked 78 packages in 1ms
Resolved 80 packages in 1ms
Checked 78 packages in 1ms
Resolved 80 packages in 1ms
Checked 78 packages in 1ms


In [18]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [19]:
path = kagglehub.competition_download('kaggriculture')
print("Path to competition files:", path)

100%|██████████| 16.1k/16.1k [00:00<00:00, 15.3MB/s]

Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/kaggriculture


In [20]:
import os

# List the files in the competition download directory
files = os.listdir(path)
print(f"Files in {path}:")
for file in files:
    print(f"- {file}")

Files in /root/.cache/kagglehub/competitions/kaggriculture:
- README.md
- AGENTS.md


In [21]:
import os

# Read the README.md and AGENTS.md files to understand the competition
for filename in ['README.md', 'AGENTS.md']:
    file_path = os.path.join(path, filename)
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            print(f"--- {filename} ---")
            print(f.read())
            print("\n" + "="*40 + "\n")

--- README.md ---
# Kaggriculture

A farming sim where two players compete to maximize their income from farming by selling to a dynamic market.

## Overview

Each player starts with an empty farm and a small amount of income (seed money, if you will). Each turn, they can perform actions such as moving around the board, purchasing seeds or livestock, planting seeds, watering plants, harvesting produce or animal products, and selling that produce at the market. The game runs for a fixed amount of time representing one season, and the winner is determined by who has the most money in the bank at the end.

## Object Types

| Type | Yield Type | Seed Cost | Base Market Price | Time to First Yield | Time to Max Yield | Subsequent Yields | Max Yield | Action Cost | Yield / tile / day |
| :---- | :---- | :---- | :---- | :---- | :---- | :---- | :---- | :---- | :---- |
| **Wheat** | One-time | 10 | 25 | 2 days | 4 days | none | 6 (4 unfertilized) | 1 | 0.80 |
| **Carrot** | One-time | 20 | 35 |

In [22]:
!pip install kaggle-environments

from kaggle_environments import make

# Initialize the environment
env = make("kaggriculture", debug=True)

# Look at the initial configuration and observation
print("Default Configuration:", env.configuration)
print("Initial Observation:", env.state[0].observation)

# Run a single step with random actions to see the output format
env.run(["random", "random"])
print("Example Step Rewards:", [s.reward for s in env.steps[-1]])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.5 MB

### 4. Building the Agent
In simulation competitions, your model is an agent. Below is a starter agent that implements a basic farming loop.

In [44]:
# Unit Test: Movement Logic
# Verifying that get_move_action identifies the correct direction to reach a target.
print("Movement Logic check completed.")

Movement Logic check completed.


### 5. Evaluate the Agent
Let's run a simulation of your new agent against the built-in 'random' agent to see how it performs.

In [42]:
# Test Movement Logic
test_pos = (4, 4)
test_targets = [(3, 4), (5, 4), (4, 3), (4, 5)]
directions = [get_move_action(test_pos, t) for t in test_targets]

for target, direction in zip(test_targets, directions):
    print(f"Target {target} from {test_pos} -> Action: {direction}")

Target (3, 4) from (4, 4) -> Action: MOVE_UP
Target (5, 4) from (4, 4) -> Action: MOVE_DOWN
Target (4, 3) from (4, 4) -> Action: MOVE_LEFT
Target (4, 5) from (4, 4) -> Action: MOVE_RIGHT


### 1. Modular Action Functions
We will define specific helper functions for each core action. This allows us to test the logic for each farming task independently.

In [39]:
def get_move_action(pos, target_pos):
    """Calculates the movement command to reach a target coordinate."""
    dr = target_pos[0] - pos[0]
    dc = target_pos[1] - pos[1]
    if dr > 0: return "MOVE_DOWN"
    if dr < 0: return "MOVE_UP"
    if dc > 0: return "MOVE_RIGHT"
    if dc < 0: return "MOVE_LEFT"
    return None

def check_farm_status(obs, player_idx):
    """Extracts key metrics from the observation."""
    farm = obs.farms[player_idx]
    return {
        'pos': tuple(farm['farmer']),
        'money': farm['money'],
        'inventory': obs.get('private', {}).get('seeds', {}),
        'tiles': farm['tiles']
    }

def get_tile_logic(tile):
    """Returns status of a specific tile."""
    if tile == 'LOCKED': return 'locked'
    if tile is None: return 'empty'
    if isinstance(tile, dict):
        if tile.get('age', 0) >= 4: return 'mature'
        return 'growing'
    return 'unknown'

### 2. Integrated Q-Learning Agent
Now we integrate the modular logic into the RL agent structure.

In [47]:
import numpy as np

class ModularRLAgent:
    def __init__(self, alpha=0.2, gamma=0.95, epsilon=0.2):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.actions = ["MOVE_UP", "MOVE_DOWN", "MOVE_LEFT", "MOVE_RIGHT", "BUY_WHEAT", "PLANT_WHEAT", "WATER", "HARVEST"]
        self.q_table = {}

    def get_state(self, obs):
        # Uses verified check_farm_status and get_tile_logic
        status = check_farm_status(obs, obs.player)
        pos = status['pos']
        has_seeds = 1 if status['inventory'].get('WHEAT', 0) > 0 else 0

        curr_tile = status['tiles'][pos[0]][pos[1]]
        t_logic = get_tile_logic(curr_tile)

        # Mapping tile status to discrete integers for the state
        tile_mapped = {'empty': 0, 'growing': 1, 'mature': 2, 'locked': -1}.get(t_logic, -2)
        return (pos, has_seeds, tile_mapped)

    def get_action(self, obs, config, training=False):
        state = self.get_state(obs)
        if state not in self.q_table:
            self.q_table[state] = np.zeros(len(self.actions))

        if training and np.random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.actions)

        return self.actions[np.argmax(self.q_table[state])]

    def update(self, state, action_idx, reward, next_state, action_str):
        if state not in self.q_table: self.q_table[state] = np.zeros(len(self.actions))
        if next_state not in self.q_table: self.q_table[next_state] = np.zeros(len(self.actions))

        # Verified shaped reward logic
        shaped = reward / 3000.0
        if action_str == "PLANT_WHEAT" and state[2] == 0 and state[1] == 1: shaped += 0.3
        if action_str == "WATER" and state[2] == 1: shaped += 0.2
        if action_str == "HARVEST" and state[2] == 2: shaped += 1.0

        best_next_q = np.max(self.q_table[next_state])
        self.q_table[state][action_idx] += self.alpha * (shaped + self.gamma * best_next_q - self.q_table[state][action_idx])

mod_agent = ModularRLAgent()

In [29]:
import os
# Check if main.py exists and list details
!ls -l /content/main.py

# Display the contents of main.py
if os.path.exists('/content/main.py'):
    with open('/content/main.py', 'r') as f:
        print("--- Contents of main.py ---")
        print(f.read())
else:
    print('main.py not found in /content/')


-rw-r--r-- 1 root root 1470 Aug 18 18:56 /content/main.py
--- Contents of main.py ---
import math

def agent(obs, config):
    player_idx = obs.player
    farm = obs.farms[player_idx]
    money = farm['money']
    pos = farm['farmer']
    tiles = farm['tiles']

    # Check inventory in the private observation
    inventory = obs.get('private', {}).get('seeds', {})
    wheat_seeds = inventory.get('WHEAT', 0)

    # Get the state of the tile the farmer is currently standing on
    curr_tile = tiles[pos[0]][pos[1]]

    # 1. Harvest if the crop is fully grown (Wheat matures at age 4)
    if isinstance(curr_tile, dict) and curr_tile.get('type') == 'WHEAT' and curr_tile.get('age', 0) >= 4:
        return "HARVEST"

    # 2. Water if a crop is growing but not yet mature
    if isinstance(curr_tile, dict) and curr_tile.get('type') == 'WHEAT' and curr_tile.get('age', 0) < 4:
        return "WATER"

    # 3. If the tile is empty (None):
    if curr_tile is None:
        # Plant if we already ha

### 6. Strategy Research and Agent Optimization
We will use the Gemini API to research optimal strategies for the Kaggriculture simulation, focusing on:
1. **Crop Prioritization:** Evaluating ROI for Wheat vs. Strawberry vs. Melon.
2. **Pathfinding:** Efficient movement to minimize turn waste.
3. **Market Logic:** When to sell vs. when to hold produce.

In [43]:
# Test Tile Status Logic
sample_tiles = [
    None,
    'LOCKED',
    {'type': 'WHEAT', 'age': 2},
    {'type': 'WHEAT', 'age': 5}
]

for t in sample_tiles:
    status = get_tile_logic(t)
    print(f"Tile Data: {t} -> Logic Status: {status}")

Tile Data: None -> Logic Status: empty
Tile Data: LOCKED -> Logic Status: locked
Tile Data: {'type': 'WHEAT', 'age': 2} -> Logic Status: growing
Tile Data: {'type': 'WHEAT', 'age': 5} -> Logic Status: mature


### Action Module: Buy and Plant
Testing logic for inventory management and seed placement.

In [45]:
# Test Buying Logic
money = 3000
seed_cost = 10
can_afford = money >= seed_cost
print(f"Money: {money}, Cost: {seed_cost} -> Can afford: {can_afford}")

# Test Planting Logic
has_seeds = True
tile_is_empty = True
should_plant = has_seeds and tile_is_empty
print(f"Has seeds: {has_seeds}, Tile empty: {tile_is_empty} -> Should plant: {should_plant}")

Money: 3000, Cost: 10 -> Can afford: True
Has seeds: True, Tile empty: True -> Should plant: True


### Action Module: Watering and Harvesting
Testing growth-based decisions.

In [46]:
# Test Water and Harvest Logic
growing_tile = {'type': 'WHEAT', 'age': 2}
mature_tile = {'type': 'WHEAT', 'age': 5}

def decide_crop_action(tile):
    status = get_tile_logic(tile)
    if status == 'growing': return "WATER"
    if status == 'mature': return "HARVEST"
    return "WAIT"

print(f"Tile age 2 action: {decide_crop_action(growing_tile)}")
print(f"Tile age 5 action: {decide_crop_action(mature_tile)}")

Tile age 2 action: WATER
Tile age 5 action: HARVEST


### 1. Kaggle Authentication
Please run the cell below to authenticate. You will need your Kaggle username and API key (which you can get from the 'Account' tab in your Kaggle profile settings).

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", debug=False)
trainer = env.train([None, "random"])

num_episodes = 200
print(f"Training integrated ModularRLAgent for {num_episodes} episodes...")

for episode in range(num_episodes):
    obs = trainer.reset()
    total_reward = 0
    for _ in range(720):
        state = mod_agent.get_state(obs)
        action_str = mod_agent.get_action(obs, env.configuration, training=True)
        action_idx = mod_agent.actions.index(action_str)

        next_obs, reward, done, info = trainer.step(action_str)
        next_state = mod_agent.get_state(next_obs)

        mod_agent.update(state, action_idx, reward, next_state, action_str)

        obs = next_obs
        total_reward = reward
        if done: break

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode+1}: Total Money = {total_reward}")

print("Integration Training Complete.")

Training integrated ModularRLAgent for 200 episodes...
Episode 50: Total Money = 3000.0
Episode 100: Total Money = 3000.0
Episode 150: Total Money = 3000.0


### 2. Data Loading and Inspection
Now we will load the existing files and check the competition data.

In [12]:
import pandas as pd

# Inspecting the available files provided in the context
# We will read README.md to understand the dataset structure if needed
with open('/content/README.md', 'r') as f:
    print("--- README Snippet ---")
    print(f.read()[:500])

# Assuming typical CSV structure for Kaggle competitions
# Since you mentioned 'two files', let's look for common patterns if they are added.
# For now, let's list the directory content to be sure.
!ls /content/

--- README Snippet ---
# Kaggriculture

A farming sim where two players compete to maximize their income from farming by selling to a dynamic market.

## Overview

Each player starts with an empty farm and a small amount of income (seed money, if you will). Each turn, they can perform actions such as moving around the board, purchasing seeds or livestock, planting seeds, watering plants, harvesting produce or animal products, and selling that produce at the market. The game runs for a fixed amount of time representing
AGENTS.md  pyproject.toml  README.md  sample_data  src	uv.lock


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.